# O topo da tabela crua não é medicina

Uma partição do FAERS, 12.000 relatórios. Antes de qualquer estatística, a
pergunta é o que o campo de reação realmente contém.

In [1]:
import duckdb

PARTITION = "../data/parquet/year=2025/quarter=1/part=0001-of-0028"

drugs = f"'{PARTITION}/report_drug.parquet'"
reactions = f"'{PARTITION}/report_reaction.parquet'"
reports = f"'{PARTITION}/report.parquet'"

In [2]:
top_terms = f"""
    SELECT reactionmeddrapt AS term, count(*) AS rows
    FROM {reactions}
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 12
"""

duckdb.sql(top_terms)

┌──────────────────────────────────────┬───────┐
│                 term                 │ rows  │
│               varchar                │ int64 │
├──────────────────────────────────────┼───────┤
│ Off label use                        │   992 │
│ Drug ineffective                     │   756 │
│ Fatigue                              │   581 │
│ Diarrhoea                            │   484 │
│ Product dose omission issue          │   473 │
│ Nausea                               │   459 │
│ Headache                             │   421 │
│ Pruritus                             │   410 │
│ Dyspnoea                             │   403 │
│ Rash                                 │   345 │
│ Product use in unapproved indication │   336 │
│ Condition aggravated                 │   336 │
└──────────────────────────────────────┴───────┘
  12 rows                            2 columns

`Off label use`, `Product use in unapproved indication`,
`Drug ineffective` — nenhum deles é algo que aconteceu com um
paciente. Registram como o produto foi usado, ou se ele funcionou.
Ordenados por desproporcionalidade ficariam no topo de toda tabela
e não significariam nada.

A lista de exclusão é a resposta: um CSV versionado, uma razão por
linha, revisado a cada milestone.

In [3]:
excluded = duckdb.sql("""
    SELECT term, category, reason
    FROM read_csv('../reference/excluded_terms.csv', comment='#')
""").df()

len(excluded), excluded.category.value_counts().to_dict()

(187,
 {'administration': 55,
  'product_quality': 37,
  'device': 28,
  'supply': 24,
  'exposure': 21,
  'efficacy': 9,
  'indication': 7,
  'therapy_decision': 5,
  'no_event': 1})

O `comment='#'` acima é estrutural. O arquivo abre com um cabeçalho em
prosa explicando a regra de inclusão, e uma leitura sem essa flag
retorna zero linhas e nenhum erro — a lista deixaria de existir em
silêncio. `analysis/prr.py` levanta erro numa leitura vazia por isso.

In [4]:
after = f"""
    SELECT reactionmeddrapt AS term, count(*) AS rows
    FROM {reactions}
    WHERE reactionmeddrapt NOT IN (
        SELECT term FROM read_csv('../reference/excluded_terms.csv', comment='#')
    )
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 12
"""

duckdb.sql(after)

┌────────────┬───────┐
│    term    │ rows  │
│  varchar   │ int64 │
├────────────┼───────┤
│ Fatigue    │   581 │
│ Diarrhoea  │   484 │
│ Nausea     │   459 │
│ Headache   │   421 │
│ Pruritus   │   410 │
│ Dyspnoea   │   403 │
│ Rash       │   345 │
│ Death      │   329 │
│ Vomiting   │   313 │
│ Pain       │   313 │
│ Arthralgia │   312 │
│ Dizziness  │   287 │
└────────────┴───────┘
  12 rows  2 columns

## 187 termos removem 15,4% das linhas de reação

O que sobra são sintomas: fadiga, diarreia, náusea, dor de cabeça.
`Death` fica em oitavo, que é o formato certo para um sistema de
reporte espontâneo — desfechos graves são reportados, e não são a
maioria do que se reporta.

A lista é um piso, não uma enumeração. Termos de procedimento
(`Chemotherapy`, `Radiotherapy`) também não são respostas do corpo,
mas excluí-los à mão é perder: precisam da hierarquia MedDRA, e o
openFDA entrega só o termo preferido.

In [5]:
duckdb.sql(f"""
    SELECT
        count(*) AS reaction_rows,
        count(*) FILTER (WHERE reactionmeddrapt IN (
            SELECT term FROM read_csv('../reference/excluded_terms.csv', comment='#')
        )) AS excluded_rows,
        count(DISTINCT reactionmeddrapt) AS distinct_terms
    FROM {reactions}
""")

┌───────────────┬───────────────┬────────────────┐
│ reaction_rows │ excluded_rows │ distinct_terms │
│     int64     │     int64     │     int64      │
├───────────────┼───────────────┼────────────────┤
│         44916 │          6900 │           4281 │
└───────────────┴───────────────┴────────────────┘